In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/train.csv')
df.columns = df.columns.str.strip()

# Suppression colonnes >80% NaN
df = df.drop(columns=['PoolQC', 'Fence', 'MiscFeature', 'Alley'], errors='ignore')

# Transformation variable cible
df['SalePrice_log'] = np.log(df['SalePrice'])
df = df.drop(columns=['SalePrice'])

# Suppression colonnes inutiles
df = df.drop(columns=['Id', 'GarageArea'], errors='ignore')

# Correction type MSSubClass
df['MSSubClass'] = df['MSSubClass'].astype(str)

df_clean = df.copy()
print(df_clean.shape)

(1460, 75)


In [2]:
missing = df_clean.isna().sum()
missing[missing > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [3]:
print(df_clean.isna().sum().sum())
print(df_clean.shape)

0
(1460, 75)


In [4]:
cat_cols = df_clean.select_dtypes(include='object').columns
print(f"Nombre de colonnes catégorielles : {len(cat_cols)}")
print(cat_cols.tolist())

Nombre de colonnes catégorielles : 43
['MSSubClass', 'MSZoning', 'LotFrontage', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'SaleType', 'SaleCondition']


In [5]:
cardinalite = df_clean.select_dtypes(include='object').nunique().sort_values(ascending=False)
print(cardinalite)

MasVnrArea       328
LotFrontage      111
GarageYrBlt       98
Neighborhood      25
Exterior2nd       16
MSSubClass        15
Exterior1st       15
SaleType           9
Condition1         9
HouseStyle         8
RoofMatl           8
Condition2         8
Functional         7
GarageType         7
BsmtFinType2       7
BsmtFinType1       7
Foundation         6
Electrical         6
FireplaceQu        6
GarageQual         6
Heating            6
GarageCond         6
SaleCondition      6
RoofStyle          6
BsmtQual           5
HeatingQC          5
LotConfig          5
BsmtCond           5
BldgType           5
MasVnrType         5
MSZoning           5
BsmtExposure       5
ExterCond          5
ExterQual          4
KitchenQual        4
GarageFinish       4
LandContour        4
LotShape           4
LandSlope          3
PavedDrive         3
CentralAir         2
Utilities          2
Street             2
dtype: int64


In [6]:
cols_numeric = ['MasVnrArea', 'LotFrontage', 'GarageYrBlt']

for col in cols_numeric:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print(df_clean[cols_numeric].dtypes)

MasVnrArea     float64
LotFrontage    float64
GarageYrBlt    float64
dtype: object


In [7]:
print(df_clean[cols_numeric].dtypes)
print(df_clean[cols_numeric].isna().sum())

MasVnrArea     float64
LotFrontage    float64
GarageYrBlt    float64
dtype: object
MasVnrArea       8
LotFrontage    259
GarageYrBlt     81
dtype: int64


In [8]:
from sklearn.impute import SimpleImputer

imputer_median = SimpleImputer(strategy='median')
df_clean[['LotFrontage', 'MasVnrArea']] = imputer_median.fit_transform(
    df_clean[['LotFrontage', 'MasVnrArea']]
)

df_clean['GarageYrBlt'] = df_clean['GarageYrBlt'].fillna(0)

print(df_clean[['LotFrontage', 'MasVnrArea', 'GarageYrBlt']].isna().sum())

LotFrontage    0
MasVnrArea     0
GarageYrBlt    0
dtype: int64


In [9]:
df_encoded = pd.get_dummies(df_clean, drop_first=True)
print(f"Shape avant encodage : {df_clean.shape}")
print(f"Shape après encodage : {df_encoded.shape}")

Shape avant encodage : (1460, 75)
Shape après encodage : (1460, 261)


In [10]:
from sklearn.preprocessing import StandardScaler

# Colonnes numériques uniquement
num_cols = df_encoded.select_dtypes(include=['int64', 'float64']).columns
num_cols = num_cols.drop('SalePrice_log')  # on ne scale pas la cible

scaler = StandardScaler()
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

print(df_encoded[num_cols].describe().round(2))

       LotFrontage  LotArea  OverallQual  OverallCond  YearBuilt  \
count      1460.00  1460.00      1460.00      1460.00    1460.00   
mean          0.00    -0.00         0.00         0.00       0.00   
std           1.00     1.00         1.00         1.00       1.00   
min          -2.22    -0.92        -3.69        -4.11      -3.29   
25%          -0.45    -0.30        -0.80        -0.52      -0.57   
50%          -0.04    -0.10        -0.07        -0.52       0.06   
75%           0.41     0.11         0.65         0.38       0.95   
max          11.04    20.52         2.82         3.08       1.28   

       YearRemodAdd  MasVnrArea  BsmtFinSF1  BsmtFinSF2  BsmtUnfSF  ...  \
count       1460.00     1460.00     1460.00     1460.00    1460.00  ...   
mean           0.00       -0.00       -0.00       -0.00      -0.00  ...   
std            1.00        1.00        1.00        1.00       1.00  ...   
min           -1.69       -0.57       -0.97       -0.29      -1.28  ...   
25%         

In [11]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['SalePrice_log'])
y = df_encoded['SalePrice_log']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

X_train : (1168, 260)
X_test  : (292, 260)
y_train : (1168,)
y_test  : (292,)


In [12]:
import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Données sauvegardées ✓")

Données sauvegardées ✓
